# Module 1: LangGraph 201 — Building a Research Agent from Scratch

> Part of the **Modular Workshops** series. Standalone, ~45 min.

We'll build the same kind of research agent Module 2 builds with Deep Agents — but here we wire it ourselves with LangGraph. By the end you'll have seen the moving parts of an agent: **state**, **tools**, **nodes**, **edges**, the prebuilt `create_agent()`, **middleware**, **multi-agent supervision**, **long-term memory**, and **human-in-the-loop** interrupts.

Four parts:

1. **Manual ReAct agent** — wire the loop yourself with `StateGraph`. Teaches state, nodes, edges.
2. **`create_agent()` + middleware** — the same agent in 5 lines, with a small middleware to customize behavior.
3. **Supervisor + subagent** — multi-agent delegation.
4. **Long-term memory + HITL** — `InMemoryStore` for cross-thread memory; `interrupt()` to confirm writes with a human.


## Setup


In [ ]:
import sys
from pathlib import Path
project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from utils.models import model
from utils.utils import show_graph
from utils.search import resilient_tavily_search

from datetime import datetime
from typing import Annotated, Literal
from typing_extensions import TypedDict

from langchain_core.tools import tool
from langchain_core.messages import AnyMessage, HumanMessage, SystemMessage, ToolMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver
from langgraph.prebuilt import ToolNode

from langsmith import uuid7

print("Ready")


## Part 1: Manual ReAct Agent

Wire the ReAct loop yourself. This is the canonical place to learn LangGraph's three primitives — **state**, **nodes**, and **edges**.

<img src="../images/react_agent.png" style="width: auto; max-height: 320px; border-radius: 8px;">


### 1.1 State

**State** is the agent's memory — a shared data structure that flows between nodes. Each node reads state and returns a partial update.

We track two fields:

- `messages` — the conversation history. The `add_messages` reducer makes updates *append* rather than replace.
- `iterations` — how many times the researcher node has run. LangGraph already caps each run at a default 25 super-steps, but tracking it ourselves shows what custom state looks like and lets us add a tighter ceiling.


In [ ]:
class ResearcherState(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]
    iterations: int  # number of times the researcher node has run


### 1.2 Tools

**Tools** are functions the LLM can call. The `@tool` decorator turns a Python function into a LangChain tool.

We keep the `@tool` def inline so the pattern stays visible; the retry + canned-fallback logic for Tavily lives in `utils/search.py` (`resilient_tavily_search`) so a flaky network doesn't kill the lesson.


In [ ]:
@tool(parse_docstring=True)
def tavily_search(query: str) -> str:
    """Search the web for information on a given query.

    Args:
        query: Search query to execute.
    """
    return resilient_tavily_search(query, max_retries=2)

researcher_tools = [tavily_search]
model_with_tools = model.bind_tools(researcher_tools)

### 1.3 Nodes

**Nodes** are just Python functions. Each takes the graph's `State`, runs some logic, and returns an update.

Two nodes:
- **`researcher`** — the LLM reasoning step. Binds tools to the model and calls it.
- **`tools`** — uses LangChain's prebuilt `ToolNode` to execute whichever tool(s) the LLM picked.


In [ ]:
researcher_prompt = f"""You are a research assistant. Today is {datetime.now().strftime('%Y-%m-%d')}.

Search the web to answer the user's question, then synthesize what you found into a brief, clear summary.
Stop calling tools once you have enough information. At most 3 search calls.
"""

def researcher(state: ResearcherState):
    response = model_with_tools.invoke(
        [SystemMessage(content=researcher_prompt)] + state["messages"]
    )
    return {"messages": [response], "iterations": state.get("iterations", 0) + 1}

tools_node = ToolNode(researcher_tools)


### 1.4 Edges

**Edges** connect the nodes and define the flow.

- **Normal edges** always go to a fixed target.
- **Conditional edges** route dynamically via a function that inspects state.

After the LLM step: if it called `research_complete` (or we've hit the iteration limit), end. Otherwise run the tool node and loop back.


In [ ]:
def should_continue(state: ResearcherState) -> Literal["tools", "__end__"]:
    last = state["messages"][-1]
    # End when the model produced a final message (no tool calls).
    if not last.tool_calls:
        return END
    if state.get("iterations", 0) >= 5:
        return END 
    return "tools"


### 1.5 Compile and run


In [ ]:
builder = StateGraph(ResearcherState)
builder.add_node("researcher", researcher)
builder.add_node("tools", tools_node)

builder.add_edge(START, "researcher")
builder.add_conditional_edges("researcher", should_continue, ["tools", END])
builder.add_edge("tools", "researcher")

manual_agent = builder.compile()
show_graph(manual_agent)


In [ ]:
result = manual_agent.invoke({
    "messages": [HumanMessage(content="What is LangGraph used for? Be brief.")],
    "iterations": 0,
})
print(result["messages"][-1].content)


## Part 2: `create_agent()` + Middleware

LangChain ships a pre-built ReAct agent — `create_agent()`. Same loop, ~5 lines.


In [ ]:
from langchain.agents import create_agent

prebuilt_agent = create_agent(
    model=model,
    tools=[tavily_search],
    system_prompt=researcher_prompt,
)

result = prebuilt_agent.invoke({
    "messages": [HumanMessage(content="What is LangGraph used for? Be brief.")],
})
print(result["messages"][-1].content)


### Middleware — customize behavior without rewriting the agent

`create_agent()` accepts **middleware**: hooks that wrap `wrap_model_call` (around every LLM call) or `wrap_tool_call` (around every tool call). Use them to inject system-prompt rules, log tool calls, intercept tool args, etc. — without touching the agent's internals.

Below: a `wrap_tool_call` middleware that logs every tool the agent invokes.


In [ ]:
from langchain.agents.middleware import wrap_tool_call

tool_log = []

@wrap_tool_call
def log_tool_calls(request, handler):
    """Log every tool call as it happens."""
    entry = {"tool": request.tool_call["name"], "args": request.tool_call["args"]}
    tool_log.append(entry)
    print(f"[tool] {entry['tool']}({entry['args']})")
    return handler(request)

agent_with_logging = create_agent(
    model=model,
    tools=[tavily_search],
    system_prompt=researcher_prompt,
    middleware=[log_tool_calls],
)

result = agent_with_logging.invoke({
    "messages": [HumanMessage(content="What is the LangChain v1 release about? Be brief.")],
})
print(result["messages"][-1].content)


## Part 3: Supervisor + Researcher Subagent

A **supervisor** decides what to research and delegates. The supervisor sees only its own conversation; the researcher runs in isolation. Module 2 gives you this for free via `task()` and `subagents=[...]` — here we build it.

The pattern: the subagent is exposed to the supervisor as a tool. The supervisor calls `conduct_research(topic=...)` and the tool invokes the researcher.

<img src="../images/supervisor.png" style="width: auto; max-height: 320px; border-radius: 8px;">


In [ ]:
# Researcher subagent: reuse the prebuilt create_agent
researcher_subagent = create_agent(
    model=model,
    tools=[tavily_search],
    system_prompt=researcher_prompt,
)

@tool
def conduct_research(topic: str) -> str:
    """Delegate a research topic to the researcher. Give one topic at a time."""
    sub_result = researcher_subagent.invoke({
        "messages": [HumanMessage(content=topic)],
    })
    return sub_result["messages"][-1].content

supervisor_prompt = """You are a research supervisor. Decompose the user's question into focused subtopics, delegate each one to the researcher via `conduct_research`, then synthesize a final answer.

Rules:
- One delegation at a time.
- After each result, decide if you need more research or you're done.
- At most 3 delegations.
"""

supervisor = create_agent(
    model=model,
    tools=[conduct_research],
    system_prompt=supervisor_prompt,
)

show_graph(supervisor, xray=True)


In [ ]:
result = supervisor.invoke({
    "messages": [HumanMessage(content="Briefly compare LangGraph and CrewAI for building agent workflows.")],
})
print(result["messages"][-1].content)


## Part 4: Long-Term Memory with Human-in-the-Loop

Extend the **research supervisor from Part 3** so it remembers user preferences about *how* to research — target audience, depth, style — across threads.

- Saving a preference goes through `interrupt()` so the human approves first.
- Two LangGraph store-access patterns side by side: a `runtime: ToolRuntime` inside a tool, and a `store: BaseStore` parameter on a node.
- Same dynamic-prompt pattern LangGraph 201's multi-agent notebook uses: a `load_*` node populates a state field, and the supervisor's system prompt formats from it on every turn.

<img src="../images/supervisor_with_memory.png" style="width: auto; max-height: 280px; border-radius: 8px;">


### 4.1 The Store

`InMemoryStore` is the simplest `BaseStore` — in-process, no persistence across restarts but survives across threads. In production you'd swap it for a backed store (Redis, Postgres, or the one LangSmith Deployments injects).


In [ ]:
from langgraph.store.memory import InMemoryStore
from langgraph.store.base import BaseStore
from langgraph.types import interrupt, Command
from langchain.tools import ToolRuntime

store = InMemoryStore()


### 4.2 Two ways to reach the Store

LangGraph gives you two natural places to read or write long-term memory:

- **From a tool** — declare a `runtime: ToolRuntime` parameter. The runtime exposes `state`, `config`, and `store`. Best when the *agent* decides to write (e.g. "save this preference").
- **From a node** — declare a `store: BaseStore` parameter on the node function. LangGraph inspects the signature and injects the store. Best when *the graph* always needs to load something before the agent runs.

We use both: `save_preference` (tool, runtime pattern, gated by `interrupt()`) and `load_preferences` (node, BaseStore parameter, runs at the start so saved memory always reaches the model).


In [ ]:
from langchain_core.tools import tool

# --- Tool: runtime pattern. The agent decides to call this. ---
@tool
def save_preference(category: str, value: str, runtime: ToolRuntime) -> str:
    """Save a user preference about how research should be conducted.

    Args:
        category: short key like 'audience' or 'depth'.
        value: the preference itself.
    """
    decision = interrupt({
        "action": "save_preference",
        "category": category,
        "value": value,
    })
    if not decision.get("approve"):
        return f"Preference NOT saved -- user declined approval for {category}."
    runtime.store.put(("preferences",), category, {"value": value})
    return f"Preference saved: {category} = {value}. Future conversations will see this."

# --- Node: BaseStore parameter pattern. The graph runs this. ---
def load_preferences(state, store: BaseStore):
    """Read all saved preferences and stash them on state for the supervisor to use."""
    items = store.search(("preferences",))
    if not items:
        return {"loaded_preferences": "(no preferences saved yet)"}
    formatted = "\n".join(f"- {it.key}: {it.value['value']}" for it in items)
    return {"loaded_preferences": formatted}


### 4.3 Wire it into the supervisor

We rebuild the supervisor as a hand-rolled `StateGraph` (same pattern as Part 1) so its system prompt can read `loaded_preferences` from state on every iteration. It reuses `conduct_research` from Part 3 and adds `save_preference` alongside.

Shape:

```
START -> load_preferences -> supervisor -> tools -> supervisor -> END
```

`load_preferences` populates `loaded_preferences` once at the top. The supervisor's prompt template formats from it. The agent loop then proceeds normally.


In [ ]:
from langgraph.prebuilt import ToolNode

class MemoryState(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]
    loaded_preferences: str

memory_tools = [conduct_research, save_preference]
model_with_tools = model.bind_tools(memory_tools)

memory_tools_node = ToolNode(memory_tools)

memory_supervisor_prompt = """You are a research supervisor. Decompose the question, delegate to `conduct_research`, then synthesize a final answer.

== SAVED PREFERENCES FOR THIS USER ==
{loaded_preferences}
== END PREFERENCES ==

YOU MUST tailor every response to these preferences. Specifically:
- Pass relevant preferences along when calling `conduct_research`. Example: include the target audience in the topic string ("Research X for an audience of <audience>").
- Synthesize your final answer in the style the preferences indicate. If audience is "non-technical executive", lead with business impact, avoid jargon, and skip implementation details.

When the user shares a NEW preference about HOW research should be done, call `save_preference` exactly once. Don't ask for confirmation -- the approval gate is handled by the system. After save_preference returns, briefly confirm what was saved.

Rules: one delegation at a time, at most 3 delegations.
"""

def memory_supervisor_node(state: MemoryState):
    prefs = state.get("loaded_preferences") or "(none loaded)"
    response = model_with_tools.invoke(
        [SystemMessage(content=memory_supervisor_prompt.format(loaded_preferences=prefs))]
        + state["messages"]
    )
    return {"messages": [response]}

def memory_should_continue(state: MemoryState) -> Literal["tools", "__end__"]:
    last = state["messages"][-1]
    return "tools" if last.tool_calls else END

memory_builder = StateGraph(MemoryState)
memory_builder.add_node("load_preferences", load_preferences)
memory_builder.add_node("supervisor", memory_supervisor_node)
memory_builder.add_node("tools", memory_tools_node)

memory_builder.add_edge(START, "load_preferences")
memory_builder.add_edge("load_preferences", "supervisor")
memory_builder.add_conditional_edges("supervisor", memory_should_continue, ["tools", END])
memory_builder.add_edge("tools", "supervisor")

memory_agent = memory_builder.compile(checkpointer=MemorySaver(), store=store)
show_graph(memory_agent)


### 4.4 Run it — share a preference, get approval, recall in a new thread

Thread A: user shares a research preference → `save_preference` interrupts for approval → we resume → supervisor confirms the save.


In [ ]:
thread_a = {"configurable": {"thread_id": str(uuid7())}}

result = memory_agent.invoke(
    {"messages": [HumanMessage(content=(
        "My research is always for a non-technical executive audience. Remember that."
    ))]},
    config=thread_a,
)

if result.get("__interrupt__"):
    print("Paused for approval:", result["__interrupt__"][0].value)
else:
    print(result["messages"][-1].content)


In [ ]:
result = memory_agent.invoke(
    Command(resume={"approve": True}),
    config=thread_a,
)
print("After approval:", result["messages"][-1].content)


Now a **new thread**. Short-term memory (checkpointer state) is fresh, but the long-term Store survives — and `load_preferences` runs first so the supervisor sees the saved preference in its system prompt without us asking for it.

Watch the supervisor pass the audience preference along to `conduct_research`, then synthesize an exec-friendly answer.


In [ ]:
thread_b = {"configurable": {"thread_id": str(uuid7())}}

result = memory_agent.invoke(
    {"messages": [HumanMessage(content="Briefly research what LangSmith is for.")]},
    config=thread_b,
)
print("New thread answer:\n")
print(result["messages"][-1].content)

# Show the tool calls so you can see the preference was forwarded
print("\nTool calls the supervisor made:")
for m in result["messages"]:
    if getattr(m, "tool_calls", None):
        for tc in m.tool_calls:
            print(f"  {tc['name']}({tc['args']})")


## Where to Go Next

You just built — by hand — most of what Module 2 gets in ~10 lines via `create_deep_agent()`. The mapping:

| What you wrote here | What Module 2 provides |
|---|---|
| `ResearcherState` + `add_messages` + `StateGraph` | Built-in agent state schema and loop |
| `create_agent(...)` + middleware | Same `create_agent()` underneath, plus a pre-built middleware stack (filesystem, planning, summarization) |
| `conduct_research` tool wrapping the subagent | `subagents=[...]` + `task()` tool |
| `InMemoryStore` + `runtime.store` + `interrupt()` in tool | `StoreBackend` + `CompositeBackend` + `interrupt_on={"tool": True}` |

**Next:**
- **Module 2 (Deep Agents)** — see the same agent built with `create_deep_agent()`, plus filesystem, persistent memory, AGENTS.md, and skills.
- **Module 4 (LangSmith)** — every invocation above was traced. Module 4 shows how to query those traces, score them with evals, and route low-scoring runs to an annotation queue.
- **Module 3 (Deploy)** — ship the supervisor as a managed service.
